In [30]:
!pip install transformers evaluate accelerate
!pip install scikit-learn
!pip install tensorboard
!pip install pathlib
!pip install matplotlib
!pip install sentencepiece

You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/mondra/git/uzh/text-speech-processing/.venv/bin/python3 -m pip install --upgrade pip' command.


In [31]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,

)

In [32]:
from pathlib import Path
# Config
MODEL = "deepset/gbert-base"
MAX_LEN = 128
SEED=42
ROOT = Path.cwd().parent
out_dir = ROOT / "runs" / f"gbert_seed{SEED}"

# Label mapping (fixed by GAHD)
ID2LABEL = {0: "not_hate", 1: "hate"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

# Hyperparameters
BATCH_SIZE=32
NUM_EPOCHS=3
LR=2e-5

In [33]:
# load dataset
from data import load_gahd

df_train, df_dev, df_test = load_gahd()

In [ ]:
# Tokenize the Dataset
from datasets import Dataset
from transformers import BertTokenizer

set_seed(SEED)
tokenizer = BertTokenizer.from_pretrained(MODEL)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)


ds_train = Dataset.from_pandas(df_train[["text", "label"]])
ds_dev = Dataset.from_pandas(df_dev[["text", "label"]])
ds_test = Dataset.from_pandas(df_test[["text", "label"]])

tokenized_train = ds_train.map(preprocess_function, batched=True)
tokenized_dev = ds_dev.map(preprocess_function, batched=True)
tokenized_test = ds_test.map(preprocess_function, batched=True)

Map: 100%|██████████| 1646/1646 [00:00<00:00, 41215.88 examples/s]


In [35]:
# Visualize a tokenized sample
ex = tokenized_train[0]
print(ex["text"])
print(ex["input_ids"])
print(tokenizer.convert_ids_to_tokens(ex["input_ids"]))

Ich denke, alles was man tun kann, ist, es zu ignorieren, so Brown.
[102, 395, 5752, 818, 1446, 556, 545, 1913, 530, 818, 215, 818, 288, 205, 16540, 549, 818, 262, 15397, 566, 103]
['[CLS]', 'Ich', 'denke', ',', 'alles', 'was', 'man', 'tun', 'kann', ',', 'ist', ',', 'es', 'zu', 'ignor', '##ieren', ',', 'so', 'Brown', '.', '[SEP]']


In [36]:
# Initialize data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [37]:
# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)

    f1_hate = f1_score(labels, preds, pos_label=1, zero_division=0)
    f1_not_hate = f1_score(labels, preds, pos_label=0, zero_division=0)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_hate": precision_score(labels, preds, zero_division=0),
        "recall_hate": recall_score(labels, preds, zero_division=0),
        "f1_hate": f1_hate,
        "f1_not_hate": f1_not_hate,
        "f1_macro": (f1_hate + f1_not_hate) / 2,
    }

In [38]:
# Model
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5121.56it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: deepset/gbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

In [39]:
# Training Arguments
training_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        save_total_limit=3,
        logging_steps=50,
        seed=SEED,
        report_to="tensorboard",
        fp16=True
    )

In [40]:
# Training
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_dev,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
 
trainer.train()

/Users/mondra/git/uzh/text-speech-processing/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Visualize Losses
import pandas as pd
import matplotlib.pyplot as plt

log = pd.DataFrame(trainer.state.log_history)
train_log = log.dropna(subset=["loss"])
eval_log = log.dropna(subset=["eval_loss"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(train_log["step"], train_log["loss"], label="train")
ax[0].plot(eval_log["step"], eval_log["eval_loss"], "o-", label="dev")
ax[0].set(xlabel="step", ylabel="loss", title="Loss")
ax[0].legend()

ax[1].plot(eval_log["step"], eval_log["eval_f1_macro"], "o-")
ax[1].set(xlabel="step", ylabel="F1 macro", title="Dev F1 macro")
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation
test_out = trainer.predict(tokenized_test, metric_key_prefix="test")
print(test_out.metrics)